In [1]:
import torch
import numpy as np

In [9]:
a = np.random.rand(3, 1)

In [12]:
(a[:, np.newaxis]* a[np.newaxis, :]).shape

(3, 3, 1)

In [2]:
from neural_networks import feedforward

In [3]:
layers = [1, 1000, 1000, 1000, 2]
activations = 'tanh'

In [4]:
net = feedforward(
    layers,
    activations,
    adapt_act=True
)

In [5]:
x = torch.rand(10, 1, device="cuda")

In [16]:
class Trainer:

    def __init__(self, model):

        self._params = model.parameters
        self._weights = model.weights
        self._bias = model.bias
        self._device = model.device
        self._zero_grad = model.zero_grad
        
    def _flat_grad(self, params='weights'):
        
        if params == 'weights':
            gen = self._weights()
        elif params == 'bias':
            gen = self._bias()
        else:
            gen = self._params()
        
        views = []
        for p in gen:
            if p.grad is None:
                view = p.new_zeros(p.numel())
            else:
                view = p.grad.view(-1)
            views.append(view)
        return torch.cat(views, 0)
        
    def _loss_weights(self, weights, losses, param=.5):
        
        _mean = []
        _max = []
        
        for loss in losses:
            loss.backward(retain_graph=True)
            grad = self._flat_grad()
            _mean.append(grad.abs().mean())
            _max.append(grad.abs().max())    
            self._zero_grad()
        del grad
        index, m = max(
            enumerate(_max),
            key=lambda f: f[1].max()
        )

        weights_new = []
        for i, w_old in enumerate(weights):
            if i != index:
                w_new = (1 - param)* w_old + param* (m/(_mean[i]* w_old))
            else:
                w_new = torch.tensor(w_old)

            weights_new.append(w_new.item())
            
        return weights_new

    def _def_lagrange_mul(self, losses, **kwargs):
            
        self.lagrange_mul = torch.nn.ParameterList(
            torch.nn.Parameter(torch.tensor(1., device=self._device)) for _ in losses
        )
        self.lagrange_mul_opt = torch.optim.Adam(
            self.lagrange_mul,
            maximize=True,
            **kwargs
        )

In [17]:
test = Trainer(net)

In [18]:
l = net(x).mean()

In [19]:
v =  net(x).pow(2).mean()

In [20]:
test._loss_weights([1., 1.], [l, v])

[1.0, 1853.848876953125]

In [21]:
test._def_lagrange_mul([l, v])

In [22]:
test.lagrange_mul

ParameterList(
    (0): Parameter containing: [torch.float32 of size  (cuda:0)]
    (1): Parameter containing: [torch.float32 of size  (cuda:0)]
)

In [26]:
def _lbfgs_direction(old_stps, old_dirs, ro):
    
    """
    Args:
        old_steps: s_k = x_k+1 - x_k
        old_dirs: y_k = grad f_k+1 - grad f_k
        ro: rho_k = 1/(s_k.dot(y_k))
    Return:
        direction: - H_k * grad f_k
    """
    
    num_old = len(old_dirs)
    
    q = flat_grad.neg()
    for i in range(num_old - 1, -1, -1):
        al[i] = old_stps[i].dot(q) * ro[i]
        q.add_(old_dirs[i], alpha=-al[i])
    
    r = torch.mul(q, H_diag)
    for i in range(num_old):
        be_i = old_dirs[i].dot(r) * ro[i]
        r.add_(old_stps[i], alpha=al[i] - be_i)
        
    return r

In [27]:
help(_lbfgs_direction)

Help on function _lbfgs_direction in module __main__:

_lbfgs_direction(old_stps, old_dirs, ro)
    Args:
        old_steps: s_k = x_k+1 - x_k
        old_dirs: y_k = grad f_k+1 - grad f_k
        ro: rho_k = 1/(s_k.dot(y_k))
    Return:
        direction: - H_k * grad f_k



In [ ]:
def func(state):
    
    while n_iter < max_iter:
        
        n_iter += 1
        state["n_iter"] += 1
    
        ############################################################
        # compute gradient descent direction
        ############################################################
        if state["n_iter"] == 1:
            d = flat_grad.neg()
            old_dirs = []
            old_stps = []
            ro = []
            H_diag = 1
        else:
            # do lbfgs update (update memory)
            y = flat_grad.sub(prev_flat_grad)
            s = d.mul(t)
            ys = y.dot(s)  # y*s
            if ys > 1e-10:
                # updating memory
                if len(old_dirs) == history_size:
                    # shift history by one (limited-memory)
                    old_dirs.pop(0)
                    old_stps.pop(0)
                    ro.pop(0)
    
                # store new direction/step
                old_dirs.append(y)
                old_stps.append(s)
                ro.append(1.0 / ys)
    
                # update scale of initial Hessian approximation
                H_diag = ys / y.dot(y)  # (y*y)
    
            # compute the approximate (L-BFGS) inverse Hessian
            # multiplied by the gradient
            num_old = len(old_dirs)
    
            if "al" not in state:
                state["al"] = [None] * history_size
            al = state["al"]
    
            # iteration in L-BFGS loop collapsed to use just one buffer
            q = flat_grad.neg()
            for i in range(num_old - 1, -1, -1):
                al[i] = old_stps[i].dot(q) * ro[i]
                q.add_(old_dirs[i], alpha=-al[i])
    
            # multiply by initial Hessian
            # r/d is the final direction
            d = r = torch.mul(q, H_diag)
            for i in range(num_old):
                be_i = old_dirs[i].dot(r) * ro[i]
                r.add_(old_stps[i], alpha=al[i] - be_i)


In [ ]:
def _minimize_bfgs(fun, x0, args=(), jac=None, callback=None,
                   gtol=1e-5, norm=np.inf, eps=_epsilon, maxiter=None,
                   disp=False, return_all=False, finite_diff_rel_step=None,
                   xrtol=0, c1=1e-4, c2=0.9, 
                   hess_inv0=None, initial_scale=False, 
                   method_bfgs="BFGS",**unknown_options):
    
    _check_unknown_options(unknown_options)
    _check_positive_definite(hess_inv0)
    retall = return_all

    x0 = asarray(x0).flatten()
    if x0.ndim == 0:
        x0.shape = (1,)
    if maxiter is None:
        maxiter = len(x0) * 200

    sf = _prepare_scalar_function(fun, x0, jac, args=args, epsilon=eps,
                                  finite_diff_rel_step=finite_diff_rel_step)

    f = sf.fun
    myfprime = sf.grad

    old_fval = f(x0)
    gfk = myfprime(x0)

    k = 0
    N = len(x0)
    I = np.eye(N, dtype=int)
    Hk = I if hess_inv0 is None else hess_inv0
    # Sets the initial step guess to dx ~ 1
    old_old_fval = old_fval + np.linalg.norm(gfk) / 2

    xk = x0
    if retall:
        allvecs = [x0]
    warnflag = 0
    gnorm = vecnorm(gfk, ord=norm)


    """ --- while loop --- """
    while (gnorm > gtol) and (k < maxiter):
        pk = -np.dot(Hk, gfk)
        try:
            alpha_k, fc, gc, old_fval, old_old_fval, gfkp1 = \
                     _line_search_wolfe12(f, myfprime, xk, pk, gfk,
                                          old_fval, old_old_fval, amin=1e-100,
                                          amax=1e100, c1=c1, c2=c2)
        except _LineSearchError:
            # Line search failed to find a better solution.
            warnflag = 2
            break

        sk = alpha_k * pk
        xkp1 = xk + sk

        if retall:
            allvecs.append(xkp1)
        
        yk = gfkp1 - gfk
        xk = xkp1
        if gfkp1 is None:
            gfkp1 = myfprime(xkp1)

        k += 1
        gnorm = vecnorm(gfk, ord=norm)
        if (gnorm <= gtol):
            break
            
        if (alpha_k*vecnorm(pk) <= xrtol*(xrtol + vecnorm(xk))):
            break

        if not np.isfinite(old_fval):
            # We correctly found +-Inf as optimal value, or something went
            # wrong.
            warnflag = 2
            break

        rhok_inv = np.dot(yk, sk)
        
        if rhok_inv == 0.:
            rhok = 1000.0
            if disp:
                msg = "Divide-by-zero encountered: rhok assumed large"
                _print_success_message_or_warn(True, msg)
        else:
            rhok = 1. / rhok_inv
        
        if method_bfgs == "BFGS":
            if initial_scale and np.allclose(Hk,np.eye(N)):
                tauk = rhok*np.dot(yk,yk)
                Hk = Hk/tauk
            Hkyk = np.matmul(Hk,yk)
            ykHkyk = np.dot(yk,Hkyk)
            hk = ykHkyk*rhok
            Hk = Hk - rhok*(Hkyk[:, np.newaxis] * sk[np.newaxis, :] + sk[:, np.newaxis] * Hkyk[np.newaxis, :]) + rhok*(1+hk)*sk[:, np.newaxis] *sk[np.newaxis, :]

        elif method_bfgs == "BFGS_scipy":
            if initial_scale and np.allclose(Hk,np.eye(N)):
                tauk = rhok*np.dot(yk,yk)
                Hk = Hk/tauk
            A1 = I - sk[:, np.newaxis] * yk[np.newaxis, :] * rhok
            A2 = I - yk[:, np.newaxis] * sk[np.newaxis, :] * rhok
            Hk = np.dot(A1, np.dot(Hk, A2)) + (rhok * sk[:, np.newaxis] *
                                                 sk[np.newaxis, :])

        elif method_bfgs == "SSBFGS_OL":
            if initial_scale and np.allclose(Hk,np.eye(N)):
                tauk = rhok*np.dot(yk,yk)
                Hk = Hk/tauk
            else:
                bk = -alpha_k*rhok*np.dot(sk,gfk)
                tauk = 1/bk
                Hk=Hk/tauk
               
            Hkyk = np.matmul(Hk,yk)
            ykHkyk = np.dot(yk,Hkyk)
            hk = ykHkyk*rhok
            Hk = Hk - rhok*(Hkyk[:, np.newaxis] * sk[np.newaxis, :] + sk[:, np.newaxis] * Hkyk[np.newaxis, :]) + rhok*(1+hk)*sk[:, np.newaxis] *sk[np.newaxis, :]

        elif method_bfgs == "SSBFGS_AB":
            if initial_scale and np.allclose(Hk,np.eye(N)):
                tauk = rhok*np.dot(yk,yk)
                Hk = Hk/tauk
            else:
                bk = -alpha_k*rhok*np.dot(sk,gfk)
                tauk = min(1,1/bk)
                Hk=Hk/tauk
               
            Hkyk = np.matmul(Hk,yk)
            ykHkyk = np.dot(yk,Hkyk)
            hk = ykHkyk*rhok
            Hk = Hk - rhok*(Hkyk[:, np.newaxis] * sk[np.newaxis, :] + sk[:, np.newaxis] * Hkyk[np.newaxis, :]) + rhok*(1+hk)*sk[:, np.newaxis] *sk[np.newaxis, :]

        elif method_bfgs == "SSBroyden1":
            if initial_scale and np.allclose(Hk,np.eye(N)):
                Hkyk = np.matmul(Hk,yk)
                ykHkyk = np.dot(yk,Hkyk)
                hk = ykHkyk*rhok
                bk = -alpha_k*rhok*np.dot(sk,gfk)
                ak = hk*bk -1
                rhokm = min(1,hk*(1-np.sqrt(np.abs(ak)/(1+ak))))
                thetakm = (rhokm-1)/ak
                thetakp = 1/rhokm
                thetakSR1 = 1/(1-bk)
                if bk==1:
                    thetak = thetakm
                elif bk > 1:
                    thetak = max(thetakm,thetakSR1)
                else:
                    thetak = min(thetakp,thetakSR1)

                tauk = hk/(1+ak*thetak)
               
            else:
                Hkyk = np.matmul(Hk,yk)
                ykHkyk = np.dot(yk,Hkyk)
                hk = ykHkyk*rhok
                bk = -alpha_k*rhok*np.dot(sk,gfk)
                ak = bk*hk -1
                rhokm = min(1,hk*(1-np.sqrt(np.abs(ak)/(1+ak))))
                thetakm = (rhokm-1)/ak
                thetakp = max(1,bk*(1+np.sqrt(np.abs(ak)/(1+ak))))
                thetakSR1 = 1/(1-bk)
                if bk==1:
                    thetak = thetakm
                elif bk > 1:
                    thetak = max(thetakm,thetakSR1)
                else:
                    thetak = min(thetakp,thetakSR1)
                rhokk = min(1,1/bk)
                sigmak = 1 + thetak*ak
                sigmaknm1 = np.abs(sigmak)**(1./(1-N))
                if thetak<=0:
                    tauk = min(rhokk*sigmaknm1,sigmak)
                else:
                    tauk = rhokk*min(sigmaknm1,1/thetak)
            vk = sk*rhok - Hkyk/ykHkyk
            phik = (1-thetak)/(1+ak*thetak)
            Hk = (Hk - Hkyk[:,np.newaxis]*Hkyk[np.newaxis,:]/ykHkyk +\
            phik*ykHkyk*vk[:,np.newaxis]*vk[np.newaxis,:])/tauk + sk[:,np.newaxis]*sk[np.newaxis,:]*rhok

        elif method_bfgs == "SSBroyden2":
            if initial_scale and np.allclose(Hk,np.eye(N)):
                Hkyk = np.matmul(Hk,yk)
                ykHkyk = np.dot(yk,Hkyk)
                hk = ykHkyk*rhok
                bk = -alpha_k*rhok*np.dot(sk,gfk)
                ak = hk*bk -1
                rhokm = min(1,hk*(1-np.sqrt(np.abs(ak)/(1+ak))))
                thetakm = (rhokm-1)/ak
                thetakp = 1/rhokm
                thetak = max(thetakm,min(thetakp,(1-bk)/bk))
                tauk = hk/(1+ak*thetak)
               
            else:
                Hkyk = np.matmul(Hk,yk)
                ykHkyk = np.dot(yk,Hkyk)
                hk = ykHkyk*rhok
                bk = -alpha_k*rhok*np.dot(sk,gfk)
                ak = bk*hk -1
                rhokm = min(1,hk*(1-np.sqrt(np.abs(ak)/(1+ak))))
                thetakm = (rhokm-1)/ak
                thetakp = 1/rhokm
                thetak = max(thetakm,min(thetakp,(1-bk)/bk))
                rhokk = min(1,1/bk)
                sigmak = 1 + thetak*ak
                sigmaknm1 = np.abs(sigmak)**(1./(1-N))
                
                if thetak<=0:
                    tauk = min(rhokk*sigmaknm1,sigmak)
                else:
                    tauk = rhokk*min(sigmaknm1,1/thetak)
                   
            vk = sk*rhok - Hkyk/ykHkyk
            phik = (1-thetak)/(1+ak*thetak)
            Hk = (Hk - Hkyk[:,np.newaxis]*Hkyk[np.newaxis,:]/ykHkyk +\
            phik*ykHkyk*vk[:,np.newaxis]*vk[np.newaxis,:])/tauk + sk[:,np.newaxis]*sk[np.newaxis,:]*rhok
            #print(ak,rhok,np.dot(sk,gfk))
        elif method_bfgs == "SSBroyden3":
            if initial_scale and np.allclose(Hk,np.eye(N)):
                Hkyk = np.matmul(Hk,yk)
                ykHkyk = np.dot(yk,Hkyk)
                hk = ykHkyk*rhok
                bk = -alpha_k*rhok*np.dot(sk,gfk)
                ak = hk*bk -1
                rhokm = min(1,hk*(1-np.sqrt(np.abs(ak)/(1+ak))))
                thetakm = (rhokm-1)/ak
                thetak = thetakm
                tauk = hk/(1+ak*thetak)
            else:
                Hkyk = np.matmul(Hk,yk)
                ykHkyk = np.dot(yk,Hkyk)
                hk = ykHkyk*rhok
                bk = -alpha_k*rhok*np.dot(sk,gfk)
                ak = bk*hk -1
                rhokm = min(1,hk*(1-np.sqrt(np.abs(ak)/(1+ak))))
                thetakm = (rhokm-1)/ak
                thetak = thetakm
                rhokk = min(1,1/bk)
                sigmak = 1 + thetak*ak
                sigmaknm1 = np.abs(sigmak)**(1./(1-N))
                
                if thetak<=0:
                    tauk = min(rhokk*sigmaknm1,sigmak)
                else:
                    tauk = rhokk*min(sigmaknm1,1/thetak)

            vk = sk*rhok - Hkyk/ykHkyk
            phik = (1-thetak)/(1+ak*thetak)
            Hk = (Hk - Hkyk[:,np.newaxis]*Hkyk[np.newaxis,:]/ykHkyk +\
            phik*ykHkyk*vk[:,np.newaxis]*vk[np.newaxis,:])/tauk + sk[:,np.newaxis]*sk[np.newaxis,:]*rhok

        gfk = gfkp1    
        intermediate_result = OptimizeResult(x=xk, fun=old_fval,nfev = sf.nfev, njev = sf.ngev, hess_inv=Hk)
        
        
        if _call_callback_maybe_halt(callback, intermediate_result):
                break
    fval = old_fval

    if warnflag == 2:
        msg = _status_message['pr_loss']
    elif k >= maxiter:
        warnflag = 1
        msg = _status_message['maxiter']
    elif np.isnan(gnorm) or np.isnan(fval) or np.isnan(xk).any():
        warnflag = 3
        msg = _status_message['nan']
    else:
        msg = _status_message['success']

    if disp:
        _print_success_message_or_warn(warnflag, msg)
        print("         Current function value: %f" % fval)
        print("         Iterations: %d" % k)
        print("         Function evaluations: %d" % sf.nfev)
        print("         Gradient evaluations: %d" % sf.ngev)

    result = OptimizeResult(fun=fval, jac=gfk, hess_inv=Hk, nfev=sf.nfev,
                            njev=sf.ngev, status=warnflag,
                            success=(warnflag == 0), message=msg, x=xk,
                            nit=k)
    if retall:
        result['allvecs'] = allvecs
    
    return result

In [24]:
a = a.flatten()
a.shape

(3,)

In [25]:
if a.ndim == 0:
    a.shape = (1,)

In [26]:
a

array([0.64038771, 0.85957852, 0.66453885])

In [ ]:
if initial_scale and np.allclose(Hk,np.eye(N)):
    tauk = rhok*np.dot(yk,yk)
    Hk = Hk/tauk
A1 = I - sk[:, np.newaxis] * yk[np.newaxis, :] * rhok
A2 = I - yk[:, np.newaxis] * sk[np.newaxis, :] * rhok
Hk = np.dot(A1, np.dot(Hk, A2)) + (rhok * sk[:, np.newaxis] *
                                     sk[np.newaxis, :])

In [35]:
M = torch.rand(1000, 1000)
M

tensor([[0.5561, 0.4327, 0.8280,  ..., 0.4111, 0.6830, 0.1486],
        [0.0738, 0.7619, 0.5711,  ..., 0.5647, 0.1315, 0.0475],
        [0.6056, 0.2726, 0.5927,  ..., 0.9314, 0.5076, 0.6240],
        ...,
        [0.1203, 0.1469, 0.2693,  ..., 0.7079, 0.6273, 0.3374],
        [0.3938, 0.4725, 0.8020,  ..., 0.9749, 0.0323, 0.6148],
        [0.3025, 0.5034, 0.4204,  ..., 0.0396, 0.4040, 0.3566]])

In [46]:
%%time

for _ in range(1000):
    M.matmul(M.T)

CPU times: user 27.8 s, sys: 456 ms, total: 28.3 s
Wall time: 3.54 s


In [48]:
torch.all(
    M.matmul(M.T) == torch.einsum("ij, kj -> ik", M, M)
)

tensor(True)

In [49]:
%%time
for _ in range(1000):
    torch.einsum("ij, kj -> ik", M, M)

CPU times: user 27.6 s, sys: 31.5 ms, total: 27.6 s
Wall time: 3.46 s


In [56]:
test = dict(d=)

SyntaxError: invalid syntax (2802234168.py, line 1)

In [54]:
test.get('d')

0

In [75]:
a = torch.rand(3).cuda()

In [78]:
b = torch.eye(5, device=a.device)

In [79]:
b

tensor([[1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1.]], device='cuda:0')

In [ ]:
    def step(self, closure):
        
        assert len(self.param_groups) == 1
        
        closure = torch.enable_grad()(closure)
        
        group = self.param_groups[0]
        lr = group["lr"]
        max_iter = group["max_iter"]
        max_eval = group["max_eval"]
        tolerance_grad = group["tolerance_grad"]
        tolerance_change = group["tolerance_change"]
        line_search_fn = group["line_search_fn"]
        
        state = self.state[self._params[0]]
        state.setdefault("func_evals", 0)
        state.setdefault("n_iter", 0)
        
        orig_loss = closure()
        loss = float(orig_loss)
        current_evals = 1
        state["func_evals"] += 1
        
        flat_grad = self._gather_flat_grad()
        opt_cond = flat_grad.abs().max() <= tolerance_grad
        
        if opt_cond:
            return orig_loss
        
        I = torch.eye(self._numel_cache, device=flat_grad.device)
        d = state.get("d")
        t = state.get("t")
        H = state.get("H")
        prev_flat_grad = state.get("prev_flat_grad")
        prev_loss = state.get("prev_loss")
        
        n_iter = 0
        while n_iter < max_iter:
            
            n_iter += 1
            state["n_iter"] += 1
            
            if state["n_iter"] == 1:
                d = flat_grad.neg()
                H = I
            else:
                
                y = flat_grad.sub(prev_flat_grad)
                s = d.mul(t)
                ys = y.dot(s)
                
                if ys > 1e-10:
                    rho = 1. / ys
                    H = ys / y.dot(y)
                
                V = I - y.outer(s)* rho
                H = V.T.matmul(H.matmul(V)) + rho* s.outer(s)
                d = H.matmul(flat_grad.neg())
                
            if prev_flat_grad is None:
                prev_flat_grad = flat_grad.clone(memory_format=torch.contiguous_format)
            else:
                prev_flat_grad.copy_(flat_grad)
            prev_loss = loss
            
            if state["n_iter"] == 1:
                t = min(1.0, 1.0 / flat_grad.abs().sum()) * lr
            else:
                t = lr
                
            # directional derivative
            gtd = flat_grad.dot(d)
            
            # directional derivative is below tolerance
            if gtd > -tolerance_change:
                break
            
            # optional line search: user function
            ls_func_evals = 0
            if line_search_fn is not None:
                # perform line search, using user function
                if line_search_fn != "strong_wolfe":
                    raise RuntimeError("only 'strong_wolfe' is supported")
                else:
                    x_init = self._clone_param()

                    def obj_func(x, t, d):
                        return self._directional_evaluate(closure, x, t, d)

                    loss, flat_grad, t, ls_func_evals = _strong_wolfe(
                        obj_func, x_init, t, d, loss, flat_grad, gtd
                    )
                self._add_grad(t, d)
                opt_cond = flat_grad.abs().max() <= tolerance_grad
            else:
                # no line search, simply move with fixed-step
                self._add_grad(t, d)
                if n_iter != max_iter:
                    # re-evaluate function only if not in last iteration
                    # the reason we do this: in a stochastic setting,
                    # no use to re-evaluate that function here
                    with torch.enable_grad():
                        loss = float(closure())
                    flat_grad = self._gather_flat_grad()
                    opt_cond = flat_grad.abs().max() <= tolerance_grad
                    ls_func_evals = 1

            # update func eval
            current_evals += ls_func_evals
            state["func_evals"] += ls_func_evals

            ############################################################
            # check conditions
            ############################################################
            if n_iter == max_iter:
                break

            if current_evals >= max_eval:
                break

            # optimal condition
            if opt_cond:
                break

            # lack of progress
            if d.mul(t).abs().max() <= tolerance_change:
                break

            if abs(loss - prev_loss) < tolerance_change:
                break

        state["d"] = d
        state["t"] = t
        state["H"] = H
        state["prev_flat_grad"] = prev_flat_grad
        state["prev_loss"] = prev_loss

        return orig_loss

In [ ]:
closure = torch.enable_grad()(closure)



def _bfgs(x0, maxiter):

    """
    Args: 
    * x0: init point
    * maxiter: maximal number of iterations
    
    Returns: 
    
    """
    if maxiter is None:
        maxiter = len(x0) * 200
    
    
    f = sf.fun
    myfprime = sf.grad
    
    old_fval = f(x0)
    gfk = myfprime(x0)

    k = 0
    N = len(x0)
    I = np.eye(N, dtype=int)
    Hk = I if hess_inv0 is None else hess_inv0
    # Sets the initial step guess to dx ~ 1
    old_old_fval = old_fval + np.linalg.norm(gfk) / 2

    xk = x0
    if retall:
        allvecs = [x0]
    warnflag = 0
    gnorm = vecnorm(gfk, ord=norm)


    """ --- while loop --- """
    while (gnorm > gtol) and (k < maxiter):
        
        pk = -np.dot(Hk, gfk)

        sk = alpha_k * pk
        xkp1 = xk + sk

        if retall:
            allvecs.append(xkp1)
        
        yk = gfkp1 - gfk
        xk = xkp1
        if gfkp1 is None:
            gfkp1 = myfprime(xkp1)

        k += 1
        gnorm = vecnorm(gfk, ord=norm)
        if (gnorm <= gtol):
            break
            
        if (alpha_k*vecnorm(pk) <= xrtol*(xrtol + vecnorm(xk))):
            break

        if not np.isfinite(old_fval):
            # We correctly found +-Inf as optimal value, or something went
            # wrong.
            warnflag = 2
            break

        rhok_inv = np.dot(yk, sk)
        
        if rhok_inv == 0.:
            rhok = 1000.0
            if disp:
                msg = "Divide-by-zero encountered: rhok assumed large"
                _print_success_message_or_warn(True, msg)
        else:
            rhok = 1. / rhok_inv
        
        if method_bfgs == "BFGS":
            if initial_scale and np.allclose(Hk,np.eye(N)):
                tauk = rhok*np.dot(yk,yk)
                Hk = Hk/tauk
            Hkyk = np.matmul(Hk,yk)
            ykHkyk = np.dot(yk,Hkyk)
            hk = ykHkyk*rhok
            Hk = Hk - rhok*(Hkyk[:, np.newaxis] * sk[np.newaxis, :] + sk[:, np.newaxis] * Hkyk[np.newaxis, :]) + rhok*(1+hk)*sk[:, np.newaxis] *sk[np.newaxis, :]

        elif method_bfgs == "BFGS_scipy":
            if initial_scale and np.allclose(Hk,np.eye(N)):
                tauk = rhok*np.dot(yk,yk)
                Hk = Hk/tauk
            A1 = I - sk[:, np.newaxis] * yk[np.newaxis, :] * rhok
            A2 = I - yk[:, np.newaxis] * sk[np.newaxis, :] * rhok
            Hk = np.dot(A1, np.dot(Hk, A2)) + (rhok * sk[:, np.newaxis] *
                                                 sk[np.newaxis, :])